# Урок 18: Защита AI-агентов с помощью криптографических квитанций

## Практическая тетрадь

В этой тетради рассмотрены четыре упражнения:

1. **Подпишите свою первую квитанцию** для вызова инструмента агента и проверьте её.
2. **Попытайтесь изменить квитанцию** и убедитесь, что проверка не проходит.
3. **Постройте цепочку из трёх квитанций** и подтвердите целостность цепочки.
4. **Обёрните вызов инструмента Microsoft Agent Framework**, чтобы каждое действие создавало квитанцию.

Все криптографические примитивы импортируются из хорошо поддерживаемых библиотек (`pynacl` для Ed25519, `jcs` для RFC 8785 canonical JSON, `hashlib` из стандартной библиотеки Python для SHA-256). Логика квитанций выполнена на обычном Python, который вы можете читать и изменять.

Запускайте ячейки по порядку. Каждый раздел краткий и самостоятельный.


## Настройка

Установите две зависимости. Обе имеют разрешительные лицензии (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Вспомогательные утилиты

Эти два помощника обрабатывают кодирование base64url (без заполнения) и хеширование SHA-256 произвольных объектов. Они позволяют остальной части блокнота сосредоточиться на самой логике квитанции.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Раздел 1: Подпишите свой первый чек

Представьте, что наш агент для **Contoso Travel** только что нашёл рейсы из Сиднея в Лос-Анджелес для клиента. Мы хотим зафиксировать этот вызов инструмента как подписанный чек, чтобы будущий аудитор мог проверить его без необходимости нам доверять.

### Шаг 1.1: Создание ключа для подписи

В производственной среде ключ для подписи агента будет храниться в модуле аппаратной безопасности (HSM), Azure Key Vault или аналогичном защищённом хранилище. Для этого урока мы сгенерируем новый ключ в памяти.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Шаг 1.2: Сборка полезной нагрузки квитанции

Полезная нагрузка содержит всё, чему мы хотим, чтобы квитанция свидетельствовала: кто действовал, какой инструмент, с какими аргументами, что вернулось, по какой политике и когда. Мы хешируем аргументы и результат, а не включаем их напрямую, чтобы квитанция не раскрывала конфиденциальное содержимое.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Шаг 1.3: Подпишите и соберите квитанцию

Три шага:

1. Канонизируйте полезную нагрузку с помощью JCS, чтобы две реализации, создающие одну и ту же логическую квитанцию, выдавали идентичные байты.
2. Хешируйте канонизированные байты с помощью SHA-256.
3. Подпишите хеш приватным ключом Ed25519.

Подпись затем прикрепляется к исходной полезной нагрузке для создания итоговой квитанции.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Шаг 1.4: Проверка квитанции

Проверка выполняет обратный процесс. Мы удаляем подпись, пересчитываем канонический хеш и проверяем подпись с помощью открытого ключа из квитанции.

Аудитор, выполняющий эту проверку, не нуждается ни в чем, кроме самой квитанции. Не нужно вызывать сервис, обращаться к каталогу ключей или иметь доверие.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Вы должны увидеть `Receipt is valid: True`. Агент создал свою первую криптографически подписанную запись аудита.


## Раздел 2: Изменение чека

Вся суть чеков в том, что они защищены от подделки. Докажем это.

Мы изменим ровно один символ в чеке и увидим, что проверка не пройдет.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Что только что произошло?

Когда мы изменили `policy_id`, изменились канонические байты. SHA-256 хэш этих байтов изменился. Подпись (которая была сделана по исходному хэшу) больше не совпадает с новым хэшем. Проверка правильно возвращает `False`.

Невозможно изменить какое-либо поле квитанции и при этом пройти проверку, если только у злоумышленника нет приватного ключа. Пока приватный ключ хранится в хранилище ключей, а публичный ключ опубликован, подделка невозможна для сокрытия.

Попробуйте сами: измените `tool_name` или `agent_id` или `timestamp` в ячейке выше и запустите снова. Каждое изменение приводит к недействительной квитанции.


## Section 3: Цепочка квитанций

Одна квитанция защищает одно действие. Большинство агентов выполняют множество действий. Чтобы сделать всю последовательность защищённой от подделок, мы связываем каждую квитанцию с предыдущей, включая хеш предыдущей квитанции в полезную нагрузку новой квитанции.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Если кто-то удалит или изменит порядок квитанций, цепочка разорвётся именно в этой точке. Проверка любой последующей квитанции не удастся, потому что её `previous_receipt_hash` больше не соответствует фактическому хешу предшествующей квитанции.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Теперь разорвите цепочку, изменив средний чек, и повторно проверьте. Подделанный чек не проходит проверку подписи, И следующий чек не проходит проверку цепочки (поскольку его `previous_receipt_hash` больше не совпадает с хэшем изменённого среднего чека).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Квитанция 0 по-прежнему проходит проверку (она не была изменена и не имеет предшественника, от которого зависит). Квитанция 1 не проходит проверку подписи, потому что мы изменили `tool_args_hash`. Квитанция 2 не проходит проверку цепочки, потому что `previous_receipt_hash` был вычислен на основе оригинальной (теперь изменённой) квитанции 1.

Даже если злоумышленник повторно подпишет изменённую квитанцию 1 (что он не может сделать без приватного ключа), несоответствие в цепочке в квитанции 2 всё равно выявит подделку. Чтобы скрыть изменение, злоумышленнику пришлось бы повторно подписать каждую квитанцию, начиная с точки изменения, что требует владения приватным ключом.


## Section 4: Обернуть вызов инструмента агента с подписанием квитанции

В реальном развертывании вы не хотите, чтобы каждый автор агента запоминал вызов `make_receipt`. Вы хотите, чтобы подписание квитанции происходило автоматически при каждом вызове инструмента.

Вот самый простой шаблон: класс-обертка, который принимает любую вызываемую функцию инструмента и возвращает версию, испускающую квитанцию. Это адаптируется к любой агентной платформе, включая Microsoft Agent Framework (`agent_framework.azure`).

Если у вас нет настроенного проекта Azure AI Foundry, локальная имитация ниже все равно демонстрирует этот шаблон.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Интеграция с Microsoft Agent Framework

Обертка `ReceiptedTool` выше не зависит от фреймворка. Чтобы использовать ее внутри агента, построенного с помощью Microsoft Agent Framework, зарегистрируйте обернутую функцию как инструмент. Эскиз (вы замените заглушку реальной регистрацией инструмента Azure AI Foundry):

```python
# Псевдокод, показывающий форму интеграции.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="Вы агент Contoso Travel ..."
#     tools=[receipted_lookup],   # обёрнутый инструмент, а не исходная функция
# )
# response = agent.run("Найдите рейсы из Сиднея в Лос-Анджелес в июне.")
#
# # После выполнения каждый вызов инструмента агентом имеет подписанную квитанцию:
# audit_chain = receipted_lookup.receipts
```

Фреймворк агента не должен знать ничего о квитанциях. Подпись квитанции обернута вокруг инструмента, а не встроена во фреймворк. Вот как добавить источник данных в существующий код агента без его переписывания.


## Резюме и дополнительное задание

Вы:

- Сгенерировали пару ключей Ed25519.
- Создали и подписали квитанцию для вызова инструмента агента.
- Проверили квитанцию офлайн, используя только открытый ключ.
- Подделали квитанцию и увидели, что проверка не прошла.
- Построили цепочку из трёх квитанций с хэш-связью.
- Подделали среднюю ссылку в цепочке и обнаружили как сбой подписи, так и сбой связки цепочки.
- Обернули функцию инструмента автоматической подписью квитанций.

**Дополнительное задание.** Расширьте схему квитанции полем `request_id` (UUID для распределённого трассирования). Обновите `make_receipt`, чтобы включить его, и подтвердите, что квитанции по-прежнему проходят проверку сквозь весь цикл. Затем измените это поле после подписания и подтвердите, что проверка не проходит. Это заставит вас понять, как каждый байт канонического кодирования влияет на подпись.

**Важное ограничение.** Квитанции доказывают ровно три вещи: атрибуцию (этот ключ подписал этот контент), целостность (контент не изменился с момента подписания) и порядок (эта квитанция появилась после той квитанции). Они НЕ доказывают, что действие агента было правильным, что политика с `policy_id` действительно была применена, или что агент соблюдал все правила. Квитанции — это фундамент. Управление — это система, которую вы строите сверху.

Прочитайте README урока снова с учётом этого ограничения. Самая распространённая ошибка команд с квитанциями — считать, что «у нас есть квитанции» значит «мы под управлением». Это не так. Квитанции делают поведение агента проверяемым. Они не делают его правильным.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
